# Muon Regime Scaling Experiment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/colab/vit_optimizer_diagnostics/muon_regime_scaling/02_muon_regime_scaling.ipynb)

첫 번째 notebook에서 old-success regime 복원이 확인되면, 이번에는 Muon과 진단량을 고정하고 외부 제어변수를 한 축씩 바꾼다.

핵심 관점은 `diagnostic_i = diagnostic_i(N, B, update_budget, seed | model, optimizer fixed)`이다.

## 0. 실험 그룹

- `regime_endpoints`: 현재 실패 조건과 과거 성공 조건을 각각 재현
- `data_scale_fixed_updates`: batch=512, seed=7, 총 update 수를 약 3900으로 맞추고 N만 10k→20k→40k
- `update_budget_at_40k`: N=40k, batch=512, seed=7에서 update budget만 증가

한 번에 여러 full run을 돌리면 오래 걸리므로 한 그룹씩 실행한다.

In [1]:
!pip -q install datasets prodigyopt tensorboard scikit-learn

%cd /content
!rm -rf deep-learning-diagnostics-and-improvement
!git clone -q https://github.com/HisameOgasahara/deep-learning-diagnostics-and-improvement.git
%cd /content/deep-learning-diagnostics-and-improvement/colab/vit_optimizer_diagnostics/muon_regime_scaling

import sys
import math
from pathlib import Path

PARENT = Path.cwd().parent
sys.path.insert(0, str(PARENT))
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import torch

from muon_regime_runner import run_muon_regime

/content
/content/deep-learning-diagnostics-and-improvement/colab/vit_optimizer_diagnostics/muon_regime_scaling


## 1. 실행할 scaling 축 선택

기본값은 가장 해석이 단순한 `update_budget_at_40k`이다. N/B/seed를 고정하고 optimization time만 바꾼다.

In [2]:
EXPERIMENT_GROUP = "update_budget_at_40k"
# 다른 선택:
# EXPERIMENT_GROUP = "data_scale_fixed_updates"
# EXPERIMENT_GROUP = "regime_endpoints"

TARGET_UPDATES = 3900

def epochs_for_updates(train_samples, batch_size, target_updates):
    steps_per_epoch = math.ceil(train_samples / batch_size)
    return math.ceil(target_updates / steps_per_epoch)

GROUPS = {
    "regime_endpoints": [
        {
            "name": "current_failed_regime_replay",
            "seed": 42,
            "train_samples": 10_000,
            "batch_size": 256,
            "epochs": 50,
            "validation_mode": "current_test_2k",
        },
        {
            "name": "old_success_regime_replay",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": 50,
            "validation_mode": "old_train_holdout_5k",
        },
    ],
    "data_scale_fixed_updates": [
        {
            "name": "N10k_B512_U3900",
            "seed": 7,
            "train_samples": 10_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(10_000, 512, TARGET_UPDATES),
            "validation_mode": "old_train_holdout_5k",
        },
        {
            "name": "N20k_B512_U3900",
            "seed": 7,
            "train_samples": 20_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(20_000, 512, TARGET_UPDATES),
            "validation_mode": "old_train_holdout_5k",
        },
        {
            "name": "N40k_B512_U3900",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(40_000, 512, TARGET_UPDATES),
            "validation_mode": "old_train_holdout_5k",
        },
    ],
    "update_budget_at_40k": [
        {
            "name": "N40k_B512_U1000",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(40_000, 512, 1000),
            "validation_mode": "old_train_holdout_5k",
        },
        {
            "name": "N40k_B512_U2000",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(40_000, 512, 2000),
            "validation_mode": "old_train_holdout_5k",
        },
        {
            "name": "N40k_B512_U3900",
            "seed": 7,
            "train_samples": 40_000,
            "batch_size": 512,
            "epochs": epochs_for_updates(40_000, 512, 3900),
            "validation_mode": "old_train_holdout_5k",
        },
    ],
}

CONFIGS = GROUPS[EXPERIMENT_GROUP]
pd.DataFrame(CONFIGS)

,name,seed,train_samples,batch_size,epochs,validation_mode
0,N40k_B512_U1000,7,40000,512,13,old_train_holdout_5k
1,N40k_B512_U2000,7,40000,512,26,old_train_holdout_5k
2,N40k_B512_U3900,7,40000,512,50,old_train_holdout_5k


## 2. 선택한 그룹 실행

각 configuration은 별도 폴더를 사용하지만 내부 `run_name`은 `muon`으로 유지한다. 그래서 optimizer 구현을 바꾸지 않고 동일 Muon을 반복 측정한다.

주의: 각 run 뒤에 representation/Hessian/Jacobian/tangent/manifold/trajectory까지 수행하므로 학습만 하는 sweep보다 오래 걸린다.

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path("/content/muon_regime_scaling_outputs") / EXPERIMENT_GROUP

results = {}
summaries = []

for config in CONFIGS:
    print("=" * 80)
    print("running:", config["name"])
    print(config)

    result = run_muon_regime(
        config=config,
        output_root=OUTPUT_ROOT,
        device=DEVICE,
        dynamics_every=10,
    )

    results[config["name"]] = result
    summaries.append(result["summary"])

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(OUTPUT_ROOT / "group_summary.csv", index=False)
summary_df

running: N40k_B512_U1000
{'name': 'N40k_B512_U1000', 'seed': 7, 'train_samples': 40000, 'batch_size': 512, 'epochs': 13, 'validation_mode': 'old_train_holdout_5k'}


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.9MB            

...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[muon   ] epoch 01/50 | train 0.3802 | val 0.4460 | 27.7s
[muon   ] epoch 02/50 | train 0.4905 | val 0.4802 | 26.9s
[muon   ] epoch 03/50 | train 0.5235 | val 0.5298 | 26.9s
[muon   ] epoch 04/50 | train 0.5420 | val 0.5268 | 26.7s
[muon   ] epoch 05/50 | train 0.5634 | val 0.5450 | 26.7s
[muon   ] epoch 06/50 | train 0.5862 | val 0.5862 | 26.7s
[muon   ] epoch 07/50 | train 0.5973 | val 0.6172 | 27.0s
[muon   ] epoch 08/50 | train 0.6136 | val 0.6076 | 26.9s
[muon   ] epoch 09/50 | train 0.6231 | val 0.6286 | 27.0s
[muon   ] epoch 10/50 | train 0.6305 | val 0.6172 | 27.1s
[muon   ] epoch 11/50 | train 0.6397 | val 0.6300 | 27.2s
[muon   ] epoch 12/50 | train 0.6521 | val 0.6326 | 26.8s
[muon   ] epoch 13/50 | train 0.6596 | val 0.6524 | 26.7s
[muon   ] epoch 14/50 | train 0.6685 | val 0.6598 | 26.8s
[muon   ] epoch 15/50 | train 0.6760 | val 0.6708 | 27.0s
[muon   ] epoch 16/50 | train 0.6850 | val 0.6726 | 27.2s
[muon   ] epoch 17/50 | train 0.6933 | val 0.6702 | 27.2s
[muon   ] epoc

/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.3373e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.24948e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.21791e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.78947e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


Hessian Lanczos: init
Hessian Lanczos: muon
Tangent kernel: muon
Input-output Jacobian: muon
Relative sharpness: muon
Manifold geometry: muon
Parameter trajectory: muon


,name,seed,train_samples,batch_size,epochs,validation_mode,steps_per_epoch,approx_updates,train_accuracy,val_accuracy,...,tangent_effective_rank,jacobian_spectral_norm,jacobian_participation_rank,relative_sharpness,mean_class_radius,mean_class_participation_dim,mean_center_axis_alignment,mean_axis_axis_alignment,empirical_dichotomy_capacity,path_to_chord_ratio
0,N40k_B512_U1000,7,40000,512,13,old_train_holdout_5k,79,1027,0.659575,0.6524,...,19.594385,8.842570,3.250969,214.855734,10.631015,13.034487,0.471110,0.314062,0.015625,1.855034
1,N40k_B512_U2000,7,40000,512,26,old_train_holdout_5k,79,2054,0.758875,0.7278,...,19.691205,8.285521,4.201765,261.884245,10.960512,17.330976,0.469741,0.245106,0.062500,1.937709
2,N40k_B512_U3900,7,40000,512,50,old_train_holdout_5k,79,3950,0.885500,0.7458,...,19.737047,10.918456,4.318753,303.131644,11.403833,21.426263,0.416313,0.224818,0.078125,2.045798


## 3. 핵심 질서변수 표

최종 accuracy 하나가 아니라 어느 scale에서 내부 상태가 함께 바뀌는지 본다.

In [4]:
core_columns = [
    "name",
    "seed",
    "train_samples",
    "batch_size",
    "epochs",
    "approx_updates",
    "train_accuracy",
    "val_accuracy",
    "penultimate_cka_to_init",
    "penultimate_linear_probe",
    "nc1",
    "knn_purity",
    "margin_mean",
    "jacobian_spectral_norm",
    "jacobian_participation_rank",
    "tangent_target_alignment",
    "hessian_min_ritz",
    "hessian_max_ritz",
    "relative_sharpness",
    "path_to_chord_ratio",
]

summary_df[core_columns]

,name,seed,train_samples,batch_size,epochs,approx_updates,train_accuracy,val_accuracy,penultimate_cka_to_init,penultimate_linear_probe,nc1,knn_purity,margin_mean,jacobian_spectral_norm,jacobian_participation_rank,tangent_target_alignment,hessian_min_ritz,hessian_max_ritz,relative_sharpness,path_to_chord_ratio
0,N40k_B512_U1000,7,40000,512,13,1027,0.659575,0.6524,0.210601,0.6620,1.646848,0.55572,0.830483,8.842570,3.250969,0.885335,-272.723619,354.293007,214.855734,1.855034
1,N40k_B512_U2000,7,40000,512,26,2054,0.758875,0.7278,0.200360,0.7318,1.627075,0.64606,1.622405,8.285521,4.201765,0.895549,-277.600472,156.426124,261.884245,1.937709
2,N40k_B512_U3900,7,40000,512,50,3950,0.885500,0.7458,0.168149,0.7558,1.618468,0.68596,2.550002,10.918456,4.318753,0.888819,-111.959498,192.397306,303.131644,2.045798


## 4. 현재 대조군까지 붙이기

과거 성공 / 현재 실패의 저장된 reference와 이번 scaling 결과를 한 표에 합친다.

In [5]:
reference = pd.read_csv("reference_results.csv")

scaling_for_compare = summary_df.rename(columns={"name": "run_id"})
comparison = pd.concat(
    [reference, scaling_for_compare],
    ignore_index=True,
    sort=False,
)

comparison.to_csv(OUTPUT_ROOT / "reference_plus_scaling.csv", index=False)
comparison

,run_id,source,commit,drive_reference,seed,train_samples,batch_size,epochs,approx_updates,validation_mode,...,steps_per_epoch,penultimate_effective_rank,ece,tangent_effective_rank,mean_class_radius,mean_class_participation_dim,mean_center_axis_alignment,mean_axis_axis_alignment,empirical_dichotomy_capacity,path_to_chord_ratio
0,old_success_muon,prior completed run,c779521402c2395d8196299f466c7bde75e7064c,https://drive.google.com/drive/folders/14k-Psa...,7,40000,512,50,3900,old_train_holdout_5k,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,current_failed_muon,current executed notebook,main current notebook,NaN,42,10000,256,50,1950,current_test_2k,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,N40k_B512_U1000,NaN,NaN,NaN,7,40000,512,13,1027,old_train_holdout_5k,...,79.0,20.802032,0.037794,19.594385,10.631015,13.034487,0.471110,0.314062,0.015625,1.855034
3,N40k_B512_U2000,NaN,NaN,NaN,7,40000,512,26,2054,old_train_holdout_5k,...,79.0,28.845045,0.036719,19.691205,10.960512,17.330976,0.469741,0.245106,0.062500,1.937709
4,N40k_B512_U3900,NaN,NaN,NaN,7,40000,512,50,3950,old_train_holdout_5k,...,79.0,35.098259,0.102147,19.737047,11.403833,21.426263,0.416313,0.224818,0.078125,2.045798


## 5. 해석 순서

1. **학습 가능성** — train/val accuracy가 어느 scale에서 회복되는가?
2. **feature movement** — CKA/effective rank가 함께 바뀌는가?
3. **task organization** — probe/kNN 상승, NC1 하락, margin 양수화가 일어나는가?
4. **function sensitivity** — Jacobian spectral norm과 participation rank가 안정화되는가?
5. **tangent geometry** — target alignment와 tangent effective rank가 변하는가?
6. **local loss geometry** — Hessian의 극단적 ± Ritz와 relative sharpness가 완화되는가?
7. **optimization path** — path/chord ratio가 어느 regime에서 달라지는가?

여러 진단량이 accuracy 회복 지점 근처에서 함께 방향을 바꾸면 단순 상관보다 훨씬 강한 regime-transition 증거가 된다.

## 6. 실험 설계상 주의

`data_scale_fixed_updates`는 update 수를 맞추지만 작은 N에서는 같은 샘플을 더 많이 반복해서 본다. 따라서 N 효과와 optimization-time을 분리하는 실험이지, information exposure까지 완전히 동일하게 만드는 실험은 아니다.

`update_budget_at_40k`가 가장 해석이 단순하므로 먼저 실행하는 것을 권장한다.

`empirical_dichotomy_capacity`는 현재 관측에서 floor에 가까웠으므로 그대로 기록하되 핵심 판정은 radius/dimension/axis geometry 쪽에 둔다.